In [ ]:
import pandas as pd
import os

data_folder = "../data"

for filename in os.listdir(data_folder):
    if filename.endswith(".csv"):
        df = pd.read_csv(os.path.join(data_folder, filename))

        print("\nDataset:", filename)
        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())
        display(df.head(3))
        

In [ ]:

# Load the larger dataset
import pandas as pd

df = pd.read_csv(
    "../data/20000-Utterances-Training-dataset-for-chatbots-virtual-assistant-Bitext-sample.csv"
)

# Check for missing values
print("Missing values:")
print(df.isnull().sum())

# Count categories and intents
print("\nNumber of categories:", df["category"].nunique())
print("Number of intents:", df["intent"].nunique())

# Show the most common intents
print("\nMost common intents:")
print(df["intent"].value_counts().head(15))

# Check for duplicate utterances
print("\nDuplicate utterances:", df["utterance"].duplicated().sum())


In [ ]:

# Count examples for every intent
intent_counts = df["intent"].value_counts()

print("Total intents:", len(intent_counts))
print("\nExamples per intent:")
print(intent_counts.to_string())

# Show the largest and smallest intent groups
print("\nLargest intent group:", intent_counts.index[0],
      "-", intent_counts.iloc[0])

print("Smallest intent group:", intent_counts.index[-1],
      "-", intent_counts.iloc[-1])

# Calculate the imbalance ratio
imbalance_ratio = intent_counts.max() / intent_counts.min()
print("\nImbalance ratio:", round(imbalance_ratio, 2))


In [ ]:

from sklearn.model_selection import train_test_split

# Input text and target labels
X = df["utterance"].astype(str)
y = df["intent"]

# First: 70% training, 30% temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# Second: divide the temporary set equally
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Training examples:", len(X_train))
print("Validation examples:", len(X_val))
print("Test examples:", len(X_test))

print("\nUnique training intents:", y_train.nunique())
print("Unique validation intents:", y_val.nunique())
print("Unique test intents:", y_test.nunique())


In [ ]:

import tensorflow as tf
from tensorflow.keras.layers import TextVectorization

# Maximum vocabulary size
max_tokens = 10000

# Maximum number of tokens per sentence
sequence_length = 40

# Create the text-processing layer
vectorizer = TextVectorization(
    max_tokens=max_tokens,
    output_mode="int",
    output_sequence_length=sequence_length
)

# Learn the vocabulary from training text only
vectorizer.adapt(X_train)

# Check the vocabulary
vocabulary = vectorizer.get_vocabulary()

print("Vocabulary size:", len(vocabulary))
print("First 20 vocabulary items:")
print(vocabulary[:20])

# Try converting one sentence
example = tf.constant(["where is my order"])
print("\nTokenized sentence:")
print(vectorizer(example).numpy())


## ENCODE

In [ ]:

# Re-create the label encoder
from tensorflow.keras.layers import StringLookup
import tensorflow as tf

label_encoder = StringLookup(
    vocabulary=sorted(y_train.unique()),
    mask_token=None,
    num_oov_indices=0
)

# Encode labels WITHOUT subtracting 1
y_train_encoded = label_encoder(
    tf.constant(y_train.to_numpy())
).numpy()

y_val_encoded = label_encoder(
    tf.constant(y_val.to_numpy())
).numpy()

y_test_encoded = label_encoder(
    tf.constant(y_test.to_numpy())
).numpy()

num_classes = len(label_encoder.get_vocabulary())

print("Number of classes:", num_classes)
print("Minimum training label:", y_train_encoded.min())
print("Maximum training label:", y_train_encoded.max())
print("Contains negative labels:", (y_train_encoded < 0).any())


In [ ]:

num_classes = len(label_encoder.get_vocabulary())

print("Number of classes:", num_classes)


In [ ]:

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

model = Sequential([
    Embedding(
        input_dim=len(vocabulary),
        output_dim=64,
        mask_zero=True
    ),
    LSTM(64),
    Dropout(0.3),
    Dense(num_classes, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.build(input_shape=(None, sequence_length))

model.summary()


In [ ]:

# Convert utterances into integer sequences
X_train_seq = vectorizer(X_train).numpy()
X_val_seq = vectorizer(X_val).numpy()
X_test_seq = vectorizer(X_test).numpy()

# Check their shapes
print("Training input shape:", X_train_seq.shape)
print("Validation input shape:", X_val_seq.shape)
print("Test input shape:", X_test_seq.shape)

print("\nTraining label shape:", y_train_encoded.shape)
print("Validation label shape:", y_val_encoded.shape)
print("Test label shape:", y_test_encoded.shape)


In [ ]:

from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

history = model.fit(
    X_train_seq,
    y_train_encoded,
    validation_data=(X_val_seq, y_val_encoded),
    epochs=10,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)


In [ ]:

test_loss, test_accuracy = model.evaluate(
    X_test_seq,
    y_test_encoded,
    verbose=1
)

print("Test loss:", round(test_loss, 4))
print("Test accuracy:", round(test_accuracy, 4))
print("Test accuracy (%):", round(test_accuracy * 100, 2), "%")


In [ ]:

from sklearn.metrics import classification_report
import numpy as np

# Predict the intent for each test utterance
y_prob = model.predict(X_test_seq, verbose=0)
y_pred = np.argmax(y_prob, axis=1)

# Get the intent names in the correct numerical order
intent_names = label_encoder.get_vocabulary()

print(classification_report(
    y_test_encoded,
    y_pred,
    labels=np.arange(num_classes),
    target_names=intent_names,
    zero_division=0,
    digits=4
))


In [ ]:

# Check for exact overlap between training and test utterances
train_texts = set(X_train.str.strip().str.lower())
test_texts = set(X_test.str.strip().str.lower())

overlap = train_texts.intersection(test_texts)

print("Unique training utterances:", len(train_texts))
print("Unique test utterances:", len(test_texts))
print("Exact overlaps:", len(overlap))

if overlap:
    print("\nExamples of overlapping utterances:")
    for text in list(overlap)[:10]:
        print("-", text)


In [ ]:

# Normalize utterances to make comparison fair
df["normalized_utterance"] = (
    df["utterance"]
    .astype(str)
    .str.strip()
    .str.lower()
)

# Find utterances that appear more than once
duplicate_rows = df[
    df.duplicated(subset="normalized_utterance", keep=False)
].sort_values("normalized_utterance")

print("Rows belonging to repeated utterances:", len(duplicate_rows))
print(
    "Number of distinct repeated utterances:",
    duplicate_rows["normalized_utterance"].nunique()
)

# Check whether repeated utterances have different intent labels
intent_counts = (
    duplicate_rows.groupby("normalized_utterance")["intent"]
    .nunique()
)

conflicting = intent_counts[intent_counts > 1]

print("Utterances assigned to multiple intents:", len(conflicting))

# Display some examples
if len(duplicate_rows) > 0:
    print("\nExamples of repeated utterances:")
    display(
        duplicate_rows[
            ["utterance", "intent"]
        ].drop_duplicates().head(20)
    )

if len(conflicting) > 0:
    print("\nExamples with conflicting intent labels:")
    display(
        duplicate_rows[
            duplicate_rows["normalized_utterance"].isin(conflicting.index)
        ][["utterance", "intent"]]
        .drop_duplicates()
        .head(20)
    )


In [ ]:

print("Rows belonging to repeated utterances:", len(duplicate_rows))
print(
    "Distinct repeated utterances:",
    duplicate_rows["normalized_utterance"].nunique()
)
print("Utterances assigned to multiple intents:", len(conflicting))


In [ ]:

from sklearn.model_selection import train_test_split

# Keep one row per normalized utterance
df_unique = df.drop_duplicates(
    subset="normalized_utterance"
).copy()

# Inputs and target labels
X = df_unique["utterance"].astype(str)
y = df_unique["intent"]

# Split: 70% training, 15% validation, 15% testing
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Unique dataset size:", len(df_unique))
print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Test samples:", len(X_test))

# Verify that normalized utterances do not cross splits
train_texts = set(X_train.str.strip().str.lower())
val_texts = set(X_val.str.strip().str.lower())
test_texts = set(X_test.str.strip().str.lower())

print("\nTrain-test overlaps:", len(train_texts & test_texts))
print("Train-validation overlaps:", len(train_texts & val_texts))
print("Validation-test overlaps:", len(val_texts & test_texts))
print("\nNumber of intents:", y.nunique())


In [ ]:

import tensorflow as tf
from tensorflow.keras.layers import TextVectorization
from tensorflow.keras.layers import StringLookup

# Text preprocessing
max_tokens = 10000
sequence_length = 40

vectorizer = TextVectorization(
    max_tokens=max_tokens,
    output_mode="int",
    output_sequence_length=sequence_length
)

# Learn vocabulary only from training utterances
vectorizer.adapt(X_train)

# Convert utterances into integer sequences
X_train_seq = vectorizer(X_train).numpy()
X_val_seq = vectorizer(X_val).numpy()
X_test_seq = vectorizer(X_test).numpy()

# Encode intent labels consistently
intent_names = sorted(y_train.unique())

label_encoder = StringLookup(
    vocabulary=intent_names,
    mask_token=None,
    num_oov_indices=0
)

y_train_encoded = label_encoder(
    tf.constant(y_train.to_numpy())
).numpy()

y_val_encoded = label_encoder(
    tf.constant(y_val.to_numpy())
).numpy()

y_test_encoded = label_encoder(
    tf.constant(y_test.to_numpy())
).numpy()

num_classes = len(intent_names)
vocabulary = vectorizer.get_vocabulary()

print("Vocabulary size:", len(vocabulary))
print("Number of intent classes:", num_classes)
print("Training shape:", X_train_seq.shape)
print("Validation shape:", X_val_seq.shape)
print("Test shape:", X_test_seq.shape)
print("Minimum encoded label:", y_train_encoded.min())
print("Maximum encoded label:", y_train_encoded.max())


In [ ]:

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dropout, Dense
from tensorflow.keras.callbacks import EarlyStopping

# Build the LSTM model
model = Sequential([
    Embedding(
        input_dim=len(vocabulary),
        output_dim=64,
        mask_zero=True
    ),
    LSTM(64),
    Dropout(0.3),
    Dense(num_classes, activation="softmax")
])

# Compile
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.build(input_shape=(None, sequence_length))
model.summary()

# Stop training if validation loss stops improving
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

# Train
history = model.fit(
    X_train_seq,
    y_train_encoded,
    validation_data=(X_val_seq, y_val_encoded),
    epochs=10,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)


In [ ]:

test_loss, test_accuracy = model.evaluate(
    X_test_seq,
    y_test_encoded,
    verbose=1
)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")
print(f"Test accuracy (%): {test_accuracy * 100:.2f}%")


In [ ]:

import numpy as np
from sklearn.metrics import classification_report

# Predict intent probabilities for the test set
y_prob = model.predict(X_test_seq, verbose=0)

# Convert probabilities to predicted class IDs
y_pred = np.argmax(y_prob, axis=1)

# Display evaluation metrics for every intent
print(
    classification_report(
        y_test_encoded,
        y_pred,
        labels=np.arange(num_classes),
        target_names=intent_names,
        zero_division=0,
        digits=4
    )
)


In [ ]:

import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(
    y_test_encoded,
    y_pred,
    labels=np.arange(num_classes)
)

fig, ax = plt.subplots(figsize=(16, 16))

display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=intent_names
)

display.plot(
    ax=ax,
    xticks_rotation=90,
    colorbar=False,
    values_format="d"
)

plt.title("LSTM Intent Classification - Confusion Matrix")
plt.tight_layout()
plt.show()


In [ ]:

import matplotlib.pyplot as plt

epochs = range(1, len(history.history["loss"]) + 1)

plt.figure(figsize=(12, 4))

# Loss curves
plt.subplot(1, 2, 1)
plt.plot(epochs, history.history["loss"], marker="o", label="Training Loss")
plt.plot(epochs, history.history["val_loss"], marker="o", label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True)

# Accuracy curves
plt.subplot(1, 2, 2)
plt.plot(epochs, history.history["accuracy"], marker="o", label="Training Accuracy")
plt.plot(epochs, history.history["val_accuracy"], marker="o", label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()


In [ ]:

import numpy as np
import tensorflow as tf

def predict_intent(text):
    # 1. Convert the input text into a sequence
    text_sequence = vectorizer(
        tf.constant([text])
    ).numpy()

    # 2. Get prediction scores for all intents
    probabilities = model.predict(
        text_sequence,
        verbose=0
    )[0]

    # 3. Find the intent with the highest score
    predicted_index = int(np.argmax(probabilities))
    predicted_intent = intent_names[predicted_index]
    confidence = float(probabilities[predicted_index])

    # 4. Display the result
    print("Customer message:", text)
    print("Predicted intent:", predicted_intent)
    print(f"Model confidence: {confidence:.2%}")

    return predicted_intent, confidence


In [ ]:

predict_intent("Where is my order?")


In [ ]:

def predict_top_intents(text, top_k=3):
    text_sequence = vectorizer(
        tf.constant([text])
    ).numpy()

    probabilities = model.predict(
        text_sequence,
        verbose=0
    )[0]

    # Get indices of the top predictions
    top_indices = np.argsort(probabilities)[-top_k:][::-1]

    print(f"\nCustomer message: {text}")
    print("-" * 45)

    for rank, index in enumerate(top_indices, start=1):
        intent = intent_names[index]
        confidence = probabilities[index]

        print(f"{rank}. {intent}: {confidence:.2%}")

    return [
        (intent_names[i], float(probabilities[i]))
        for i in top_indices
    ]


In [ ]:
predict_top_intents("Where is my order?")

In [ ]:
predict_top_intents("I need help with my money")

In [ ]:

test_messages = [
    "I was charged twice for my order",
    "I want to get a refund",
    "How can I check my refund policy?",
    "I cannot complete my payment"
]

for message in test_messages:
    predict_top_intents(message)



In [ ]:

intents_to_inspect = [
    "payment_issue",
    "check_invoices",
    "check_refund_policy",
    "track_refund",
    "get_refund"
]

for intent in intents_to_inspect:
    print(f"\n{'=' * 60}")
    print(f"INTENT: {intent}")
    print("=" * 60)

    examples = df.loc[df["intent"] == intent, "utterance"].head(8)

    for i, utterance in enumerate(examples, start=1):
        print(f"{i}. {utterance}")


In [ ]:

diagnostic_messages = [
    "My payment failed",
    "I cannot pay for my order",
    "I want to see my invoice",
    "Where can I find my bill?",
    "What is your refund policy?",
    "Has my refund been processed?",
    "I want a refund for my cancelled order"
]

for message in diagnostic_messages:
    predict_top_intents(message, top_k=3)


In [ ]:

# Search for payment-related training examples
payment_examples = df[
    df["intent"] == "payment_issue"
]["utterance"]

keywords = ["cannot pay", "can't pay", "payment failed", "unable to pay"]

for keyword in keywords:
    matches = payment_examples[
        payment_examples.str.contains(
            keyword, case=False, regex=False
        )
    ]

    print(f"\nKeyword: {keyword}")
    print("Matching examples:", len(matches))

    for example in matches.head(5):
        print("-", example)


In [37]:

while True:
    user_message = input("\nEnter a customer message (or type 'exit' to stop): ")

    if user_message.lower().strip() == "exit":
        print("Prediction system closed.")
        break

    if not user_message.strip():
        print("Please enter a message.")
        continue

    predict_top_intents(user_message, top_k=3)



Customer message: I can't pay
---------------------------------------------
1. payment_issue: 38.49%
2. get_invoice: 10.59%
3. check_invoices: 8.91%

Customer message: I was charged twice
---------------------------------------------
1. payment_issue: 87.45%
2. get_invoice: 3.17%
3. registration_problems: 3.12%

Customer message: I have a problem with my payment
---------------------------------------------
1. payment_issue: 99.52%
2. registration_problems: 0.20%
3. get_invoice: 0.13%

Customer message: I cannot complete my order payment
---------------------------------------------
1. track_order: 73.12%
2. change_order: 14.16%
3. check_invoices: 4.36%
Prediction system closed.


In [38]:

import pandas as pd

history_df = pd.DataFrame(history.history)

print(history_df.tail(10))


   accuracy      loss  val_accuracy  val_loss
0  0.719503  1.100278      0.956463  0.200670
1  0.976716  0.123638      0.988863  0.052923
2  0.994794  0.038979      0.994600  0.026421
3  0.997686  0.020691      0.995613  0.017752
4  0.998988  0.011553      0.993588  0.019368
5  0.999566  0.006464      0.993925  0.018698


In [39]:

# New customer messages for generalization testing
new_sentences = [
    "My card keeps getting declined at checkout",
    "The website won't let me finish paying",
    "I paid yesterday but my order is still missing",
    "Can I change the size of an item I bought?",
    "I accidentally ordered two of the same product",
    "The delivery person never arrived",
    "I received the wrong item in my package",
    "Someone accessed my account without permission",
    "Please stop sending promotional emails to me",
    "I want my money back because the product is broken",
    "Why was I charged more than the listed price?",
    "I forgot which email I used to sign up"
]

for sentence in new_sentences:
    predict_top_intents(sentence, top_k=3)
    print()



Customer message: My card keeps getting declined at checkout
---------------------------------------------
1. payment_issue: 91.58%
2. track_order: 3.90%
3. check_invoices: 3.32%


Customer message: The website won't let me finish paying
---------------------------------------------
1. payment_issue: 75.89%
2. track_order: 8.94%
3. place_order: 8.29%


Customer message: I paid yesterday but my order is still missing
---------------------------------------------
1. track_order: 97.84%
2. payment_issue: 1.09%
3. place_order: 0.32%


Customer message: Can I change the size of an item I bought?
---------------------------------------------
1. place_order: 42.94%
2. change_order: 24.53%
3. complaint: 14.43%


Customer message: I accidentally ordered two of the same product
---------------------------------------------
1. create_account: 60.92%
2. place_order: 22.22%
3. contact_customer_service: 8.31%


Customer message: The delivery person never arrived
------------------------------------

In [41]:

import pandas as pd
import numpy as np
import tensorflow as tf

results = []

for sentence in new_sentences:
    sequence = vectorizer(tf.constant([sentence])).numpy()
    probabilities = model.predict(sequence, verbose=0)[0]

    top_indices = np.argsort(probabilities)[-3:][::-1]

    results.append({
        "Sentence": sentence,
        "Top prediction": intent_names[top_indices[0]],
        "Confidence": f"{probabilities[top_indices[0]]:.2%}",
        "Second prediction": intent_names[top_indices[1]],
        "Second confidence": f"{probabilities[top_indices[1]]:.2%}",
        "Third prediction": intent_names[top_indices[2]],
        "Third confidence": f"{probabilities[top_indices[2]]:.2%}"
    })

results_df = pd.DataFrame(results)
from IPython.display import display as show_table

show_table(results_df)

,Sentence,Top prediction,Confidence,Second prediction,Second confidence,Third prediction,Third confidence
0,My card keeps getting declined at checkout,payment_issue,91.58%,track_order,3.90%,check_invoices,3.32%
1,The website won't let me finish paying,payment_issue,75.89%,track_order,8.94%,place_order,8.29%
2,I paid yesterday but my order is still missing,track_order,97.84%,payment_issue,1.09%,place_order,0.32%
3,Can I change the size of an item I bought?,place_order,42.94%,change_order,24.53%,complaint,14.43%
4,I accidentally ordered two of the same product,create_account,60.92%,place_order,22.22%,contact_customer_service,8.31%
5,The delivery person never arrived,contact_human_agent,48.49%,delivery_period,26.27%,delivery_options,13.35%
6,I received the wrong item in my package,track_order,78.15%,change_order,10.48%,track_refund,5.79%
7,Someone accessed my account without permission,recover_password,62.21%,delete_account,33.04%,contact_human_agent,1.53%
8,Please stop sending promotional emails to me,payment_issue,17.97%,track_order,12.41%,get_invoice,12.35%
9,I want my money back because the product is br...,check_refund_policy,75.63%,recover_password,17.66%,track_refund,1.81%
